# 🐖 CERDOS_24 — Auditoría de errores de YOLO26
**FaroPigSeg | Auditoría descriptiva en test, sin entrenamiento**

**Objetivo:** identificar imágenes críticas, sesgos del conteo, diferencia entre semillas, distribución de confianza y posibles máscaras duplicadas. Se usan **predicciones congeladas del experimento 23** y su Excel asociado.

> ⚠️ **Rigor metodológico:** este notebook **NO optimiza umbrales con test** ni modifica pesos. La superposición de máscaras es un **indicador exploratorio**, no equivale automáticamente a un falso positivo. El análisis no constituye validación externa.

**Instrucciones:** ejecutar las celdas en orden en Google Colab; cargar `CERDOS_23_YOLO26_Evaluacion_Resultados.zip` cuando lo pida (también puede buscarse en Drive). La carpeta `Resultados/CERDOS_24` se crea automáticamente y los cuadros tabulares se guardan en **Excel**.

In [ ]:
!pip -q install pandas openpyxl xlsxwriter pycocotools matplotlib seaborn

In [ ]:
import os, io, json, zipfile, hashlib, glob, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from pycocotools import mask as mask_utils
from IPython.display import display

try:
    from google.colab import drive, files
    IN_COLAB=True
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    IN_COLAB=False

PROJECT_NAME='Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖'
DRIVE_ROOT=Path('/content/drive/MyDrive') if IN_COLAB else Path.cwd()
PROJECT_ROOT=DRIVE_ROOT/PROJECT_NAME
OUT=PROJECT_ROOT/'03_Resultados'/'Resultados'/'CERDOS_24'
OUT.mkdir(parents=True,exist_ok=True)
print('Carpeta de salida:',OUT)


## 1. Localizar resultados congelados del experimento 23
Si está en otra carpeta de Drive, el buscador lo localizará. Si no aparece, Colab solicitará cargarlo.

In [ ]:
ZIP_NAME='CERDOS_23_YOLO26_Evaluacion_Resultados.zip'
candidates=[Path('/content')/ZIP_NAME,Path.cwd()/ZIP_NAME]
if PROJECT_ROOT.exists():
    candidates.extend(PROJECT_ROOT.rglob(ZIP_NAME))
    candidates.extend(DRIVE_ROOT.glob('**/'+ZIP_NAME)) if not any(p.exists() for p in candidates) else None
zip_path=next((p for p in candidates if p.exists()),None)
if zip_path is None and IN_COLAB:
    print('Sube el ZIP de evaluación del experimento 23:')
    up=files.upload()
    if ZIP_NAME not in up: raise FileNotFoundError('Debes cargar '+ZIP_NAME)
    zip_path=Path('/content')/ZIP_NAME
if zip_path is None: raise FileNotFoundError('No se encuentra '+ZIP_NAME)
print('Fuente:',zip_path)
with zipfile.ZipFile(zip_path) as z:
    names=z.namelist()
    required=['CERDOS_23_YOLO26_Evaluacion_COCO_3_Semillas.xlsx','CERDOS_23_Protocolo.json']
    assert all(x in names for x in required), 'ZIP incompleto o incorrecto'
    report=json.loads(z.read('CERDOS_23_Informe_Evaluacion.json'))
    protocol=json.loads(z.read('CERDOS_23_Protocolo.json'))
    cnt=pd.read_excel(io.BytesIO(z.read(required[0])),sheet_name='Conteo_Por_Imagen')
    cnt['semilla']=cnt['semilla'].astype(int)
    cnt['image_id']=cnt['image_id'].astype(int)
    for field in ['gt_conteo','pred_conteo_025','error_firmado_pred_menos_gt']:
        cnt[field]=pd.to_numeric(cnt[field],errors='raise')
    prediction_files={seed:f'CERDOS_23_Pred_seed_{seed}.json' for seed in protocol['seeds']}
    assert all(fn in names for fn in prediction_files.values()),'Faltan predicciones COCO en el ZIP'
    predictions={int(seed):json.loads(z.read(fn)) for seed,fn in prediction_files.items()}
print('Imágenes:',cnt.image_id.nunique(),' | Filas:',len(cnt),' | Semillas:',list(predictions))
assert cnt.image_id.nunique()==protocol['test_images']
assert cnt.groupby('semilla').size().eq(protocol['test_images']).all()
assert cnt.groupby('semilla')['gt_conteo'].sum().eq(protocol['test_gt_instances']).all()
display(cnt.head())


## 2. Comprobaciones de integridad y consistencia del conteo

In [ ]:
assert not cnt.duplicated(['semilla','image_id']).any(),'Duplicados semilla-imagen'
assert ((cnt.pred_conteo_025-cnt.gt_conteo)==cnt.error_firmado_pred_menos_gt).all(),'Errores de conteo inconsistentes'
for seed, preds in predictions.items():
    counts=defaultdict(int)
    for p in preds:
        assert 'score' in p and 'segmentation' in p and 'image_id' in p
        if float(p['score'])>=float(protocol['count_conf_threshold']):
            counts[int(p['image_id'])]+=1
    sub=cnt[cnt.semilla==seed].copy()
    compare=sub['image_id'].map(counts).fillna(0).astype(int).to_numpy()
    assert np.array_equal(compare,sub['pred_conteo_025'].astype(int).to_numpy()),f'Conteo no coincide con predicciones para semilla {seed}'
print('✅ Conteo regenerado desde los JSON y verificado para las tres semillas.')


## 3. Resumen global y desglose por fecha

In [ ]:
cnt['error_abs']=cnt.error_firmado_pred_menos_gt.abs()
cnt['conteo_exacto']=(cnt.error_abs==0).astype(int)
summary=(cnt.groupby('semilla').agg(imagenes=('image_id','size'),mae=('error_abs','mean'),
        sesgo=('error_firmado_pred_menos_gt','mean'),rmse=('error_firmado_pred_menos_gt',lambda x:float(np.sqrt(np.mean(x**2)))),
        mediana_error_abs=('error_abs','median'),conteo_exacto_pct=('conteo_exacto',lambda x:100*x.mean()),
        subconteo_pct=('error_firmado_pred_menos_gt',lambda x:100*(x<0).mean()),
        sobreconteo_pct=('error_firmado_pred_menos_gt',lambda x:100*(x>0).mean())).reset_index())
by_date=(cnt.groupby(['semilla','fecha']).agg(imagenes=('image_id','size'),mae=('error_abs','mean'),
          sesgo=('error_firmado_pred_menos_gt','mean'),conteo_exacto_pct=('conteo_exacto',lambda x:100*x.mean()),
          gt_promedio=('gt_conteo','mean'),pred_promedio=('pred_conteo_025','mean')).reset_index())
display(summary.round(3));display(by_date.round(3))


## 4. Imágenes críticas y diferencias entre semillas

In [ ]:
worst=(cnt.sort_values(['semilla','error_abs'],ascending=[True,False]).groupby('semilla').head(25).copy())
wide=cnt.pivot(index='image_id',columns='semilla',values='pred_conteo_025')
wide.columns=[f'pred_{x}' for x in wide.columns]
base=cnt.drop_duplicates('image_id')[['image_id','fecha','imagen','gt_conteo']].set_index('image_id')
disagreement=base.join(wide)
pred_cols=[x for x in disagreement if x.startswith('pred_')]
disagreement['amplitud_semillas']=disagreement[pred_cols].max(axis=1)-disagreement[pred_cols].min(axis=1)
disagreement['error_abs_promedio']=disagreement[pred_cols].sub(disagreement['gt_conteo'],axis=0).abs().mean(axis=1)
disagreement=disagreement.sort_values(['amplitud_semillas','error_abs_promedio'],ascending=False).reset_index()
display(worst[['semilla','fecha','imagen','gt_conteo','pred_conteo_025','error_firmado_pred_menos_gt']].groupby('semilla').head(5))
display(disagreement.head(12))


## 5. Auditoría de máscaras solapadas (exploratoria)
Calcula IoU entre parejas de máscaras **solo por encima del umbral fijo 0.25**. Usa el área del RLE para una aproximación eficiente y después compara las máscaras. Se marca `IoU ≥ 0.70` como *solapamiento alto*, **no** como duplicación demostrada. En casos con contacto físico u oclusión la superposición también puede ser legítima. El valor 0.70 solo sirve para diagnosticar y no se optimiza.

In [ ]:
CONF_FIJO=float(protocol['count_conf_threshold'])
IOU_DIAGNOSTICO=0.70
MIN_SCORE_TO_AUDIT=CONF_FIJO
rows=[]
for seed,preds in predictions.items():
    grouped=defaultdict(list)
    for p in preds:
        if float(p['score'])>=MIN_SCORE_TO_AUDIT:
            grouped[int(p['image_id'])].append(p)
    for image_id,prs in grouped.items():
        if len(prs)<2: continue
        rles=[]
        for p in prs:
            seg=p['segmentation']; cnts=seg['counts']
            rles.append({'size':seg['size'],'counts':cnts.encode('ascii') if isinstance(cnts,str) else cnts})
        areas=np.asarray(mask_utils.area(rles),dtype=float)
        # COCO API compara todas las parejas (mismas clases); diagonal ignorada
        ious=mask_utils.iou(rles,rles,[0]*len(rles))
        for i in range(len(prs)):
            for j in range(i+1,len(prs)):
                value=float(ious[i,j])
                if value>=IOU_DIAGNOSTICO:
                    rows.append({'semilla':seed,'image_id':image_id,'mascara_i':i,'mascara_j':j,
                        'iou_mascaras':value,'score_i':float(prs[i]['score']),'score_j':float(prs[j]['score']),
                        'area_i_px':float(areas[i]),'area_j_px':float(areas[j])})
overlaps=pd.DataFrame(rows,columns=['semilla','image_id','mascara_i','mascara_j','iou_mascaras','score_i','score_j','area_i_px','area_j_px'])
overlap_counts=(overlaps.groupby(['semilla','image_id']).size().rename('pares_solapados').reset_index()
                  if not overlaps.empty else pd.DataFrame(columns=['semilla','image_id','pares_solapados']))
audit=cnt.merge(overlap_counts,on=['semilla','image_id'],how='left')
audit['pares_solapados']=audit['pares_solapados'].fillna(0).astype(int)
print('Parejas de máscaras con IoU ≥',IOU_DIAGNOSTICO,':',len(overlaps))
display(audit.sort_values(['pares_solapados','error_abs'],ascending=False)[['semilla','image_id','fecha','gt_conteo','pred_conteo_025','error_abs','pares_solapados']].head(20))


## 6. Gráficos de diagnóstico (PNG)

In [ ]:
sns.set_theme(style='whitegrid',context='notebook')
fig,ax=plt.subplots(figsize=(9,5))
sns.boxplot(data=cnt,x='semilla',y='error_firmado_pred_menos_gt',ax=ax)
ax.axhline(0,color='black',linewidth=1);ax.set(xlabel='Semilla',ylabel='Predicho − real',title='Distribución del error de conteo (test)')
fig.tight_layout();fig.savefig(OUT/'24_error_conteo_semillas.png',dpi=190);plt.show()
fig,ax=plt.subplots(figsize=(11,5))
sns.barplot(data=by_date,x='fecha',y='mae',hue='semilla',ax=ax)
ax.set(xlabel='Fecha de adquisición',ylabel='MAE de conteo',title='Error por fecha y semilla')
plt.xticks(rotation=25);fig.tight_layout();fig.savefig(OUT/'24_mae_por_fecha.png',dpi=190);plt.show()
fig,ax=plt.subplots(figsize=(9,5))
sns.scatterplot(data=audit,x='gt_conteo',y='pred_conteo_025',hue='semilla',alpha=.35,ax=ax)
lims=[0,max(audit.gt_conteo.max(),audit.pred_conteo_025.max())+1]
ax.plot(lims,lims,'k--',linewidth=1);ax.set(xlim=lims,ylim=lims,xlabel='Conteo real',ylabel='Conteo predicho',title='Conteos por imagen')
fig.tight_layout();fig.savefig(OUT/'24_conteo_real_vs_predicho.png',dpi=190);plt.show()


## 7. Exportar Excel, protocolo, informe y ZIP
Se guarda un Excel con resultados globales, por fecha, imágenes críticas, discrepancias y posibles solapamientos. También se redacta un informe JSON reproducible.

In [ ]:
excel=OUT/'CERDOS_24_Auditoria_Errores_YOLO26.xlsx'
with pd.ExcelWriter(excel,engine='xlsxwriter') as writer:
    for name,df in [('Resumen_Global',summary),('Resumen_Fecha',by_date),('Por_Imagen',audit),
                    ('Top_Errores',worst),('Discrepancia_Semillas',disagreement),('Solapes_IoU_070',overlaps)]:
        df.to_excel(writer,sheet_name=name,index=False)
        ws=writer.sheets[name]
        ws.freeze_panes(1,0);ws.autofilter(0,0,len(df),max(len(df.columns)-1,0))
        for idx,c in enumerate(df.columns):
            ws.set_column(idx,idx,min(max(len(str(c))+3,15),58))
report24={'estudio':'CERDOS_24_Auditoria_descriptiva_test','sin_entrenamiento':True,
 'fuente_zip':zip_path.name,'sha256_zip':hashlib.sha256(zip_path.read_bytes()).hexdigest(),
 'semillas':list(map(int,predictions.keys())),'umbral_conteo_fijo':CONF_FIJO,
 'umbral_solapamiento_exploratorio':IOU_DIAGNOSTICO,'imagenes_unicas':int(cnt.image_id.nunique()),
 'resumen_global':summary.to_dict('records'),
 'observacion':'NO es validacion externa. Los solapes altos no demuestran detecciones duplicadas ni errores sin revisar GT.'}
(OUT/'CERDOS_24_Informe.json').write_text(json.dumps(report24,ensure_ascii=False,indent=2),encoding='utf-8')
archive=OUT/'CERDOS_24_Resultados.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as zz:
    for p in [excel,OUT/'CERDOS_24_Informe.json',*OUT.glob('24_*.png')]:zz.write(p,p.name)
print('✅ Archivos guardados en:',OUT)
for p in [excel,archive]: print(' •',p.name,round(p.stat().st_size/1024,1),'KB')
if IN_COLAB:
    print('Puedes descargar ahora el ZIP:')
    files.download(str(archive))


## 8. Interpretación y siguiente experimento
- Un **sesgo positivo** confirma sobreconteo promedio, pero no identifica su mecanismo por sí solo.
- Si las imágenes con mayor sobreconteo también presentan múltiples pares con IoU alto, **inspeccionar visualmente** esas máscaras y las anotaciones. No declararlas falsos positivos automáticamente.
- Las diferencias por fecha son descriptivas: es necesario comprobar iluminación, densidad y oclusiones antes de explicar las causas.
- Para cualquier mejora futura del umbral se necesita un **protocolo en validación**, con evaluación final aislada.
- Próximo paso sugerido: **CERDOS_25** sobre calibración y selección del umbral usando solo el conjunto de validación.